# リーグの進捗率別の得点数

シーズンがどれくらい進んだか（進捗率）を6つに分けて、区分ごとの得点数を数える。
シーズンの序盤・中盤・終盤で、得点の入りやすさが変わるかを見る。

### 進捗率
`survival_provisional_point_gap` と同じ定義にする。

進捗率 = そのチームが何試合目か ÷ そのチームのそのシーズンの全試合数

- 全試合数は、データにある試合数を使う。一部の試合しか入っていないシーズンでは、入っている試合の中での進み具合になる。
- 例：38試合のシーズンの19試合目は 19 / 38 = 0.5（50%）
- 得点は、**得点したチームの進捗率**で分類する。

### 集計するもの
| 項目 | 内容 |
|---|---|
| 得点数 | その区分で入った得点の数 |
| チーム試合数 | その区分に入るチームの試合の数（1試合につきホームとアウェイで2つ） |
| 1チーム1試合あたり得点 | 得点数 ÷ チーム試合数 |

区分によって試合の数が違うので、得点数だけでなく、1試合あたりも見る。

## 1. データの読み込み

`statsbomb_data` にある全リーグ・全シーズンの `*_goals_and_red_cards.csv` を読み込んで、1つの表にまとめる。

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

# グラフの日本語が文字化けしないようにする
plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "Yu Gothic", "Meiryo", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
DATA_DIR = "../../statsbomb_data"

csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_goals_and_red_cards.csv"), recursive=True))
print(f"{len(csv_paths)} 件のファイルが見つかりました")

df_all = pd.concat([pd.read_csv(p) for p in csv_paths], ignore_index=True)
print(f"行数: {len(df_all)}, 試合数: {df_all['match_id'].nunique()}")

延長戦まで行った試合は、90分の試合とは別の扱いにしたいので除く。試合終了の時計が115分を超える試合を延長戦とみなす。

In [ ]:
end_rows = df_all[df_all["red_card"].isna() & df_all["home_away"].isna() & df_all["end_time_minutes"].notna()]
extra_time_ids = end_rows.loc[end_rows["end_time_minutes"] > 115, "match_id"]
print("除く試合:", list(extra_time_ids))

df_all = df_all[~df_all["match_id"].isin(extra_time_ids)]
print(f"試合数: {df_all['match_id'].nunique()}")

## 2. 進捗率を計算する

チームごとに試合を日付順に並べて、何試合目かを数える。それをそのシーズンの全試合数で割って、進捗率を出す。

In [ ]:
# 試合ごとの情報（1試合1行）
matches_meta = (
    df_all[["league_code", "season_name", "match_id", "match_date", "home_team", "away_team"]]
    .drop_duplicates(subset=["league_code", "season_name", "match_id"])
)

# ホームとアウェイを縦に並べて、チームごとの表にする
TEAM_COLS = ["league_code", "season_name", "match_id", "match_date", "team", "side"]
home_df = matches_meta.rename(columns={"home_team": "team"}).assign(side="home")[TEAM_COLS]
away_df = matches_meta.rename(columns={"away_team": "team"}).assign(side="away")[TEAM_COLS]
team_matches = pd.concat([home_df, away_df], ignore_index=True)

# チームごとに日付順に並べて、何試合目かを数える
team_matches = team_matches.sort_values(["league_code", "season_name", "team", "match_date", "match_id"], kind="mergesort")
group_cols = ["league_code", "season_name", "team"]
team_matches["team_match_number"] = team_matches.groupby(group_cols).cumcount() + 1
team_matches["team_total_matches"] = team_matches.groupby(group_cols)["team"].transform("size")

# 進捗率 = 何試合目か ÷ 全試合数
# 小数第6位で丸める（区分の境目にちょうどのる試合が、計算の誤差で隣の区分に入らないようにするため）
team_matches["progress"] = (team_matches["team_match_number"] / team_matches["team_total_matches"]).round(6)

print(f"チームの試合数（ホームとアウェイを別々に数える）: {len(team_matches)}")
team_matches[["league_code", "season_name", "team", "team_match_number", "team_total_matches", "progress"]].head()

## 3. 進捗率を6つに分ける

0〜1を同じ幅で分けると、区分ごとのチーム試合数がそろわない。シーズンの試合数が38試合や22試合など何通りもあり、7で割り切れない分の端数が、いつも最初の区分に少なく、最後の区分に多く出るため。

そこで、**区分ごとのチーム試合数がほぼ同じになるように、区切りの位置をデータから決める**。

1. 進捗率の値ごとに、チーム試合数を数えて、小さい方から足していく（累計）
2. 累計が全体の 1/6, 2/6, …, 5/6 にいちばん近くなる進捗率を、区切りにする

同じ進捗率の試合は同じ区分に入れるので、区切りはぴったりの位置にならず、少しずれる。
区切りの位置は進捗率の値そのものなので、区切りにちょうどのる試合は下側の区分に入る。

In [ ]:
N_BINS = 6

# 進捗率の値ごとに、チーム試合数を数える
progress_counts = team_matches["progress"].value_counts().sort_index()
cumulative = progress_counts.cumsum()   # その進捗率までの累計
total = cumulative.iloc[-1]

# 累計が全体の 1/6, 2/6, ... にいちばん近い進捗率を、区切りにする
inner_edges = []
for j in range(1, N_BINS):
    target = total * j / N_BINS
    nearest = (cumulative - target).abs().idxmin()
    inner_edges.append(nearest)

BIN_EDGES = np.array([0.0] + inner_edges + [1.0])
BIN_ORDER = [f"{int(round(BIN_EDGES[i] * 100))}〜{int(round(BIN_EDGES[i + 1] * 100))}%"
             for i in range(N_BINS)]

team_matches["progress_bin"] = pd.cut(team_matches["progress"], bins=BIN_EDGES, labels=BIN_ORDER, include_lowest=True)

print(f"進捗率{N_BINS}区分:")
for label in BIN_ORDER:
    print(" -", label)
print(team_matches["progress_bin"].value_counts(sort=False).to_string())

## 4. 得点を集計する

得点の行だけを取り出して、得点したチームの進捗率の区分をつける。
得点の行は、`red_card` が空で `home_away` がある行（0がホーム、1がアウェイ）。

In [ ]:
goal_rows = df_all[df_all["red_card"].isna() & df_all["home_away"].notna()].copy()
goal_rows["side"] = np.where(goal_rows["home_away"] == 0, "home", "away")

# 得点したチームの進捗率の区分をつける
goal_rows = goal_rows.merge(
    team_matches[["league_code", "season_name", "match_id", "side", "progress", "progress_bin"]],
    on=["league_code", "season_name", "match_id", "side"], how="left",
)

print(f"得点: {len(goal_rows)} 件（区分がつかなかった得点: {int(goal_rows['progress_bin'].isna().sum())} 件）")

In [ ]:
goals_by_bin = goal_rows.groupby("progress_bin", observed=False).size()
matches_by_bin = team_matches.groupby("progress_bin", observed=False).size()

summary = pd.DataFrame({"得点数": goals_by_bin, "チーム試合数": matches_by_bin})
summary["1チーム1試合あたり得点"] = (summary["得点数"] / summary["チーム試合数"]).round(3)
summary

ホームとアウェイに分けても集計する。

In [ ]:
goals_side = goal_rows.groupby(["progress_bin", "side"], observed=False).size().unstack(fill_value=0)
matches_side = team_matches.groupby(["progress_bin", "side"], observed=False).size().unstack()

summary_side = pd.DataFrame({
    "ホーム得点数": goals_side["home"],
    "アウェイ得点数": goals_side["away"],
    "ホーム1試合あたり": (goals_side["home"] / matches_side["home"]).round(3),
    "アウェイ1試合あたり": (goals_side["away"] / matches_side["away"]).round(3),
})
summary_side

## 5. 集計が正しいか確かめる

次の3つを確認する。

1. 区分ごとの得点数の合計 = 得点の総数
2. 区分ごとのチーム試合数の合計 = 試合数 × 2
3. 区分ごとのホーム得点数とアウェイ得点数の合計 = 得点数

In [ ]:
n_matches = df_all["match_id"].nunique()

print("1. 得点数の合計:", int(summary["得点数"].sum()) == len(goal_rows), f"({int(summary['得点数'].sum())} vs {len(goal_rows)})")
print("2. チーム試合数の合計:", int(summary["チーム試合数"].sum()) == n_matches * 2, f"({int(summary['チーム試合数'].sum())} vs {n_matches} x 2)")
print("3. ホーム + アウェイ = 得点数:", bool(((summary_side["ホーム得点数"] + summary_side["アウェイ得点数"]) == summary["得点数"]).all()))

## 6. グラフにする

左は区分ごとの得点数、右は1チーム1試合あたりの得点。
右のグラフの誤差棒は、得点の数がポアソン分布に従うと考えたときの95%区間。
棒の上の数字は、その区分の1チーム1試合あたり得点（誤差棒の上に書いた）。


In [ ]:
BAR_COLOR = "#6699CC"

# 1チーム1試合あたり得点の95%区間（ポアソン分布）
goals = summary["得点数"].to_numpy()
n_team_matches = summary["チーム試合数"].to_numpy()
lower = chi2.ppf(0.025, 2 * goals) / 2 / n_team_matches
upper = chi2.ppf(0.975, 2 * (goals + 1)) / 2 / n_team_matches
per_match = goals / n_team_matches

x = np.arange(N_BINS)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左：得点数
axes[0].bar(x, goals, color=BAR_COLOR, edgecolor="white")
for xi, g in zip(x, goals):
    axes[0].text(xi, g, f"{g}", ha="center", va="bottom", fontsize=10)
axes[0].set_ylabel("得点数")
axes[0].set_title("区分ごとの得点数")

# 右：1チーム1試合あたり得点
axes[1].bar(x, per_match, color=BAR_COLOR, edgecolor="white")
axes[1].errorbar(x, per_match, yerr=[per_match - lower, upper - per_match], fmt="none", ecolor="black", capsize=4)
for xi, v, u in zip(x, per_match, upper):   # 誤差棒の上に値を書く
    axes[1].text(xi, u, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
axes[1].set_ylabel("1チーム1試合あたり得点")
axes[1].set_title("区分ごとの1チーム1試合あたり得点（95%区間つき）")
axes[1].set_ylim(0, upper.max() * 1.1)

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(BIN_ORDER)
    ax.set_xlabel("シーズンの進捗率")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("リーグの進捗率別の得点", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])

os.makedirs("./figures", exist_ok=True)
out_path = "./figures/scores_by_progress.png"
fig.savefig(out_path, dpi=150)
print(f"保存しました: {out_path}")
plt.show()

## 7. 1試合あたりの得点数（両チームの合計）

ここまでは「1チームの1試合あたり」だった。ここからは、1試合そのものの得点数（ホームの得点 + アウェイの得点）を、進捗率の区分ごとに集計する。

1試合を1つの区分に入れるために、試合の進捗率を「ホームの進捗率とアウェイの進捗率の平均」とする。
ほとんどの試合では2チームの進捗率は同じ（同じ節に試合をするため）で、ずれても最大0.2くらい。区切りは、3節で決めたものをそのまま使う。

In [ ]:
# 試合ごとの進捗率 = ホームとアウェイの進捗率の平均
match_df = team_matches.pivot_table(
    index=["league_code", "season_name", "match_id"], columns="side", values="progress"
).reset_index()
match_df["match_progress"] = ((match_df["home"] + match_df["away"]) / 2).round(6)
match_df["progress_bin"] = pd.cut(match_df["match_progress"], bins=BIN_EDGES, labels=BIN_ORDER, include_lowest=True)

# 試合ごとの得点数（得点がなかった試合は0）
match_df["home_goals"] = match_df["match_id"].map(goal_rows[goal_rows["side"] == "home"].groupby("match_id").size()).fillna(0).astype(int)
match_df["away_goals"] = match_df["match_id"].map(goal_rows[goal_rows["side"] == "away"].groupby("match_id").size()).fillna(0).astype(int)
match_df["total_goals"] = match_df["home_goals"] + match_df["away_goals"]

print(f"試合数: {len(match_df)}, 得点の合計: {match_df['total_goals'].sum()}")
match_df.head()

区分ごとに、試合数、得点の合計、1試合あたりの得点を求める。

In [ ]:
by_bin = match_df.groupby("progress_bin", observed=False)

match_summary = pd.DataFrame({
    "試合数": by_bin.size(),
    "得点の合計": by_bin["total_goals"].sum(),
    "ホームの得点": by_bin["home_goals"].sum(),
    "アウェイの得点": by_bin["away_goals"].sum(),
})
match_summary["1試合あたり得点"] = (match_summary["得点の合計"] / match_summary["試合数"]).round(3)
match_summary["ホーム1試合あたり"] = (match_summary["ホームの得点"] / match_summary["試合数"]).round(3)
match_summary["アウェイ1試合あたり"] = (match_summary["アウェイの得点"] / match_summary["試合数"]).round(3)
match_summary

### ホームとアウェイを分けない表

上の表からホームとアウェイの列を除いて、両チームを合わせた全得点数だけにする。一番下の「全体」の行は、全区分の合計。
1試合あたり得点の下限と上限は、95%区間（ポアソン分布）。

In [ ]:
match_total = match_summary[["試合数", "得点の合計"]].rename(columns={"得点の合計": "全得点数"})
match_total.index = match_total.index.astype(str)
match_total.loc["全体"] = match_total.sum()   # 全体の行 = 各区分の合計

match_total["1試合あたり得点"] = match_total["全得点数"] / match_total["試合数"]
match_total["下限(95%)"] = chi2.ppf(0.025, 2 * match_total["全得点数"]) / 2 / match_total["試合数"]
match_total["上限(95%)"] = chi2.ppf(0.975, 2 * (match_total["全得点数"] + 1)) / 2 / match_total["試合数"]
match_total.round(3)

1試合の得点数が0点、1点、2点、…の試合が、それぞれ区分の中で何%あるかも見る（5点以上は1つにまとめる）。

In [ ]:
goal_count = match_df["total_goals"].clip(upper=5)   # 5点以上は5にする
goal_count_share = pd.crosstab(match_df["progress_bin"], goal_count, normalize="index")
goal_count_share = goal_count_share.reindex(columns=range(6), fill_value=0)
goal_count_share.columns = ["0点", "1点", "2点", "3点", "4点", "5点以上"]
(goal_count_share * 100).round(1)

確かめる：試合数の合計は全試合数、得点の合計は得点の総数と一致するはず。

In [ ]:
print("試合数の合計:", int(match_summary["試合数"].sum()) == n_matches, f"({int(match_summary['試合数'].sum())} vs {n_matches})")
print("得点の合計:", int(match_summary["得点の合計"].sum()) == len(goal_rows), f"({int(match_summary['得点の合計'].sum())} vs {len(goal_rows)})")

## 8. グラフにする

左は区分ごとの1試合あたり得点、右は1試合の得点数（0点〜5点以上）の割合。
右のグラフは、各帯に割合（%）を書いて、隣の棒の同じ得点数の境目を点線でつないだ。
左の誤差棒は、得点の合計がポアソン分布に従うと考えたときの95%区間。
左のグラフの棒の上の数字は、その区分の1試合あたり得点（誤差棒の上に書いた）。


In [ ]:
# 1試合あたり得点の95%区間（ポアソン分布）
goals_total = match_summary["得点の合計"].to_numpy()
n_match = match_summary["試合数"].to_numpy()
lower = chi2.ppf(0.025, 2 * goals_total) / 2 / n_match
upper = chi2.ppf(0.975, 2 * (goals_total + 1)) / 2 / n_match
per_match_total = goals_total / n_match

x = np.arange(N_BINS)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左：1試合あたり得点
axes[0].bar(x, per_match_total, color=BAR_COLOR, edgecolor="white")
axes[0].errorbar(x, per_match_total, yerr=[per_match_total - lower, upper - per_match_total], fmt="none", ecolor="black", capsize=4)
for xi, v, u in zip(x, per_match_total, upper):   # 誤差棒の上に値を書く
    axes[0].text(xi, u, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
for xi, n in zip(x, n_match):
    axes[0].text(xi, 0.05, f"n={n}", ha="center", va="bottom", color="white", fontsize=10)
axes[0].set_ylabel("1試合あたり得点（ホーム + アウェイ）")
axes[0].set_title("区分ごとの1試合あたり得点（95%区間つき）")
axes[0].set_ylim(0, upper.max() * 1.1)

# 右：1試合の得点数の割合（積み上げ）
SHARE_COLORS = ["#DEEBF7", "#C6DBEF", "#9ECAE1", "#6BAED6", "#3182BD", "#08519C"]
BAR_WIDTH = 0.7
share_pct = goal_count_share.to_numpy() * 100   # 行：区分、列：0点〜5点以上（%）
cum_pct = share_pct.cumsum(axis=1)              # 下から積み上げた高さ

for k, (col, color) in enumerate(zip(goal_count_share.columns, SHARE_COLORS)):
    axes[1].bar(x, share_pct[:, k], bottom=cum_pct[:, k] - share_pct[:, k], width=BAR_WIDTH,
                color=color, edgecolor="white", label=col)

    # 各帯の真ん中に割合（%）を書く（濃い色の帯は白い文字にする）
    text_color = "white" if k >= 4 else "black"
    for j in range(N_BINS):
        axes[1].text(x[j], cum_pct[j, k] - share_pct[j, k] / 2, f"{share_pct[j, k]:.1f}%",
                     ha="center", va="center", fontsize=9, color=text_color)

    # 隣の棒の同じ得点数の境目を点線でつなぐ（一番上の境目はどれも100%なので除く）
    if k < len(goal_count_share.columns) - 1:
        for j in range(N_BINS - 1):
            axes[1].plot([x[j] + BAR_WIDTH / 2, x[j + 1] - BAR_WIDTH / 2], [cum_pct[j, k], cum_pct[j + 1, k]],
                         color="#555555", linestyle=":", linewidth=1.2)
axes[1].set_ylabel("試合の割合（%）")
axes[1].set_title("1試合の得点数の割合")
axes[1].legend(title="1試合の得点数", loc="upper left", bbox_to_anchor=(1.01, 1.0))

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(BIN_ORDER)
    ax.set_xlabel("シーズンの進捗率")

fig.suptitle("リーグの進捗率別の1試合あたり得点", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])

out_path = "./figures/scores_per_match_by_progress.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"保存しました: {out_path}")
plt.show()

## 9. 最初の区分と最後の区分で、得点数に差があるか（検定）

区分1（序盤）と区分6（終盤）で、1試合あたりの得点に差があるかを検定する。

### 記号
- $n_i$：区分 $i$ の試合数（$i=1,\dots,6$）
- $G_{ij}$：区分 $i$ の $j$ 番目のサンプル（$j=1,\dots,2n_i$）。1試合からホームの得点とアウェイの得点の2つが取れるので、サンプルは $2n_i$ 個
- $S_i=\sum_{j=1}^{2n_i}G_{ij}$：区分 $i$ の得点の合計

### 仮定
$G_{ij}$ は互いに独立で、$G_{ij}\sim\mathrm{Poisson}(\mu_{ij})$ とする。期待値 $\mu_{ij}$ は、チームごと・試合ごとに違ってよい。
独立なポアソン分布の和はポアソン分布なので、

$$S_i\sim\mathrm{Poisson}(2n_i\bar\mu_i),\qquad \bar\mu_i=\frac{1}{2n_i}\sum_{j=1}^{2n_i}\mu_{ij}$$

となる。$\bar\mu_i$ は、区分 $i$ の期待得点の平均。

### 仮説
$$H_0:\ \bar\mu_1=\bar\mu_6,\qquad H_1:\ \bar\mu_1<\bar\mu_6$$

### 検定
$S=S_1+S_6$ を固定すると、

$$S_6\mid S\ \sim\ \mathrm{Bin}(S,\,p),\qquad p=\frac{n_6\bar\mu_6}{n_1\bar\mu_1+n_6\bar\mu_6}$$

$H_0$ のもとでは $p_0=\dfrac{n_6}{n_1+n_6}$。だから検定は、

$$H_0:\ p=p_0,\qquad H_1:\ p>p_0$$

の二項検定になる（片側の p値 $=\sum_{k=s_6}^{S}\binom{S}{k}p_0^k(1-p_0)^{S-k}$）。

### 効果の大きさ
率比 $R=\bar\mu_6/\bar\mu_1$ を、$\hat R=\dfrac{S_6/(2n_6)}{S_1/(2n_1)}$ で推定する。
信頼区間は、$p$ の Clopper-Pearson 区間 $[L_p,U_p]$ を $R=\dfrac{p}{1-p}\cdot\dfrac{n_1}{n_6}$ に代入して作る。

（参考：Przyborowski & Wilenski, 1940, *Biometrika* 31, 313–323 / Krishnamoorthy & Thomson, 2004, *J. Stat. Plan. Inference* 119, 23–35）

まず、区分1と区分6の試合数 $n_1, n_6$ と、得点の合計 $S_1, S_6$ を取り出す。

In [ ]:
from scipy.stats import binomtest, norm

def add_stars(p):
    # p値に印をつける：10%未満で *、5%未満で **、1%未満で ***
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""


first_bin = BIN_ORDER[0]    # 区分1（序盤）
last_bin = BIN_ORDER[-1]    # 区分6（終盤）

n1 = int(match_summary.loc[first_bin, "試合数"])
n6 = int(match_summary.loc[last_bin, "試合数"])
S1 = int(match_summary.loc[first_bin, "得点の合計"])
S6 = int(match_summary.loc[last_bin, "得点の合計"])
S = S1 + S6

print(f"区分1 ({first_bin}): 試合数 {n1}, 得点の合計 {S1}, 1試合あたり {S1 / n1:.3f}")
print(f"区分6 ({last_bin}): 試合数 {n6}, 得点の合計 {S6}, 1試合あたり {S6 / n6:.3f}")

$H_0$ のもとでの $p_0=n_6/(n_1+n_6)$ を求めて、二項検定をする。片側（区分6のほうが多い）と両側の両方を出す。

In [ ]:
p0 = n6 / (n1 + n6)

# 片側検定（H1: 区分6のほうが得点が多い）と両側検定
p_greater = binomtest(S6, S, p0, alternative="greater").pvalue
result_two = binomtest(S6, S, p0, alternative="two-sided")
p_two = result_two.pvalue

# 正規近似のz値（参考）
z = (S6 - S * p0) / np.sqrt(S * p0 * (1 - p0))

print(f"p0 = {p0:.4f}")
print(f"H0 のもとでの S6 の期待値 = {S * p0:.1f}（実際は {S6}）")
print(f"z = {z:.3f}")
print(f"片側 p値 = {p_greater:.4f} {add_stars(p_greater)}")
print(f"両側 p値 = {p_two:.4f} {add_stars(p_two)}")
print("* : 10%有意, ** : 5%有意, *** : 1%有意")

率比 $R=\bar\mu_6/\bar\mu_1$ と、その95%信頼区間を求める。

In [ ]:
def p_to_ratio(p):
    # p を率比 R に直す：R = p / (1 - p) * n1 / n6
    return p / (1 - p) * n1 / n6

p_hat = S6 / S
ci = result_two.proportion_ci(confidence_level=0.95, method="exact")   # Clopper-Pearson

ratio_hat = p_to_ratio(p_hat)
ratio_low = p_to_ratio(ci.low)
ratio_high = p_to_ratio(ci.high)

print(f"率比 R = {ratio_hat:.3f}  （95%信頼区間 {ratio_low:.3f} 〜 {ratio_high:.3f}）")

結果を表にまとめる。p値には、有意水準10%なら `*`、5%なら `**`、1%なら `***` をつける。

In [ ]:
test_result = pd.DataFrame({
    "値": [n1, n6, S1, S6, round(S1 / n1, 4), round(S6 / n6, 4),
           round(ratio_hat, 4), round(ratio_low, 4), round(ratio_high, 4), round(z, 4),
           f"{p_greater:.4f} {add_stars(p_greater)}", f"{p_two:.4f} {add_stars(p_two)}"]
}, index=[
    "区分1の試合数", "区分6の試合数", "区分1の得点の合計", "区分6の得点の合計",
    "区分1の1試合あたり得点", "区分6の1試合あたり得点",
    "率比 R（区分6 / 区分1）", "Rの下限(95%)", "Rの上限(95%)",
    "z値（正規近似）", "p値（片側）", "p値（両側）",
])
test_result

確かめる：二項検定の片側 p値を、二項分布の確率を足し合わせる式で直接計算して、`binomtest` の結果と合うかを見る。

In [ ]:
from scipy.stats import binom

# P(S6 >= 観測値) = 1 - P(S6 <= 観測値 - 1)
p_direct = 1 - binom.cdf(S6 - 1, S, p0)
print("片側 p値（直接計算）:", round(p_direct, 6), " binomtest:", round(p_greater, 6))
print("一致:", bool(np.isclose(p_direct, p_greater)))

### 注意
- この検定は、$G_{ij}$ が独立なポアソン分布に従うという仮定に頼っている。同じ試合のホームとアウェイの得点に正の相関があると、$S_i$ のばらつきが仮定より大きくなり、p値が小さく出すぎる。
- 「区分1と区分6を比べる」という選び方は、グラフを見てから決めると多重比較の問題が出る。事前に決めた比較として扱うか、片側と両側の両方を報告するのが安全。

### 9.2 隣り合う区分の比較

同じ検定を、隣り合う区分（区分1と2、2と3、…、5と6）の5組にも行う。
区分 $i$ と区分 $i+1$ について、

$$H_0:\ \bar\mu_i=\bar\mu_{i+1},\qquad H_1:\ \bar\mu_i<\bar\mu_{i+1}$$

を、9の最初と同じ二項検定（$S_{i+1}\mid S_i+S_{i+1}\sim\mathrm{Bin}(S_i+S_{i+1},\,p)$、$H_0$ では $p_0=\dfrac{n_{i+1}}{n_i+n_{i+1}}$）で調べる。

同じデータで5回検定するので、「少なくとも1つ誤って有意になる確率」が5%より大きくなる。そこで、p値を5倍したもの（ボンフェローニ補正、1を超えたら1）も並べる。

In [ ]:
adjacent_rows = []
for i in range(N_BINS - 1):
    bin_a = BIN_ORDER[i]        # 前の区分
    bin_b = BIN_ORDER[i + 1]    # 後の区分

    n_a = int(match_summary.loc[bin_a, "試合数"])
    n_b = int(match_summary.loc[bin_b, "試合数"])
    S_a = int(match_summary.loc[bin_a, "得点の合計"])
    S_b = int(match_summary.loc[bin_b, "得点の合計"])
    S_ab = S_a + S_b
    p0_ab = n_b / (n_a + n_b)

    p_one = binomtest(S_b, S_ab, p0_ab, alternative="greater").pvalue   # 片側（後のほうが多い）
    res_ab = binomtest(S_b, S_ab, p0_ab)                                # 両側
    p_both = res_ab.pvalue
    p_both_adj = min(1.0, p_both * (N_BINS - 1))                        # ボンフェローニ補正

    # 率比 R = (後の区分の平均) / (前の区分の平均) と95%信頼区間
    ci_ab = res_ab.proportion_ci(confidence_level=0.95, method="exact")
    ratio_ab = (S_b / S_ab) / (1 - S_b / S_ab) * n_a / n_b
    ratio_ab_low = ci_ab.low / (1 - ci_ab.low) * n_a / n_b
    ratio_ab_high = ci_ab.high / (1 - ci_ab.high) * n_a / n_b

    adjacent_rows.append({
        "比較": f"{bin_a} → {bin_b}",
        "前の1試合あたり": round(S_a / n_a, 3),
        "後の1試合あたり": round(S_b / n_b, 3),
        "率比 R": round(ratio_ab, 3),
        "Rの95%区間": f"{ratio_ab_low:.3f} 〜 {ratio_ab_high:.3f}",
        "p値（片側）": f"{p_one:.4f} {add_stars(p_one)}",
        "p値（両側）": f"{p_both:.4f} {add_stars(p_both)}",
        "p値（両側・補正後）": f"{p_both_adj:.4f} {add_stars(p_both_adj)}",
    })

adjacent_result = pd.DataFrame(adjacent_rows).set_index("比較")
adjacent_result

`*` は10%有意、`**` は5%有意、`***` は1%有意。
片側は「後の区分のほうが得点が多い」という向きだけを見ているので、得点が減っている組では p値が大きくなる。

### 9.3 すべての組み合わせ

区分 $i$ と区分 $k$（$i<k$）のすべての組み合わせ（6区分なので15組）で、同じ検定を行う。

$$H_0:\ \bar\mu_i=\bar\mu_k,\qquad H_1:\ \bar\mu_i<\bar\mu_k$$

15回検定するので、ボンフェローニ補正では p値を15倍する（1を超えたら1）。補正は厳しめなので、ホルム法（p値の小さい順に $15,14,13,\dots$ 倍して、順番が逆転しないようにそろえる）も並べる。

In [ ]:
from itertools import combinations

pair_rows = []
for i, k in combinations(range(N_BINS), 2):
    bin_a = BIN_ORDER[i]    # 前の区分
    bin_b = BIN_ORDER[k]    # 後の区分

    n_a = int(match_summary.loc[bin_a, "試合数"])
    n_b = int(match_summary.loc[bin_b, "試合数"])
    S_a = int(match_summary.loc[bin_a, "得点の合計"])
    S_b = int(match_summary.loc[bin_b, "得点の合計"])
    S_ab = S_a + S_b
    p0_ab = n_b / (n_a + n_b)

    p_one = binomtest(S_b, S_ab, p0_ab, alternative="greater").pvalue   # 片側（後のほうが多い）
    res_ab = binomtest(S_b, S_ab, p0_ab)                                # 両側
    ci_ab = res_ab.proportion_ci(confidence_level=0.95, method="exact")

    pair_rows.append({
        "前": bin_a, "後": bin_b, "i": i + 1, "k": k + 1,
        "前の1試合あたり": S_a / n_a,
        "後の1試合あたり": S_b / n_b,
        "率比 R": (S_b / S_ab) / (1 - S_b / S_ab) * n_a / n_b,
        "Rの下限": ci_ab.low / (1 - ci_ab.low) * n_a / n_b,
        "Rの上限": ci_ab.high / (1 - ci_ab.high) * n_a / n_b,
        "p_片側": p_one,
        "p_両側": res_ab.pvalue,
    })

pairs = pd.DataFrame(pair_rows)
n_pairs = len(pairs)

# 補正：ボンフェローニ（15倍）とホルム（小さい順に 15, 14, ... 倍して、大小の順が崩れないようにする）
pairs["p_両側_ボンフェローニ"] = (pairs["p_両側"] * n_pairs).clip(upper=1)
order = pairs["p_両側"].sort_values().index
holm = []
running_max = 0
for rank, idx in enumerate(order):
    adjusted = min(1.0, pairs.loc[idx, "p_両側"] * (n_pairs - rank))
    running_max = max(running_max, adjusted)
    holm.append(running_max)
pairs.loc[order, "p_両側_ホルム"] = holm

print(f"{n_pairs} 組を検定した")

15組の結果を一覧にする。p値には `*`（10%）、`**`（5%）、`***`（1%）をつける。

In [ ]:
pairs_show = pd.DataFrame({
    "比較": pairs["前"] + " → " + pairs["後"],
    "前の1試合あたり": pairs["前の1試合あたり"].round(3),
    "後の1試合あたり": pairs["後の1試合あたり"].round(3),
    "率比 R": pairs["率比 R"].round(3),
    "Rの95%区間": pairs["Rの下限"].round(3).astype(str) + " 〜 " + pairs["Rの上限"].round(3).astype(str),
    "p値（片側）": [f"{p:.4f} {add_stars(p)}" for p in pairs["p_片側"]],
    "p値（両側）": [f"{p:.4f} {add_stars(p)}" for p in pairs["p_両側"]],
    "両側・ボンフェローニ": [f"{p:.4f} {add_stars(p)}" for p in pairs["p_両側_ボンフェローニ"]],
    "両側・ホルム": [f"{p:.4f} {add_stars(p)}" for p in pairs["p_両側_ホルム"]],
}).set_index("比較")
pairs_show

片側の p値を、行が前の区分、列が後の区分の表にする（$H_1$：後の区分のほうが得点が多い）。

In [ ]:
matrix = pd.DataFrame("", index=BIN_ORDER, columns=BIN_ORDER)
for _, row in pairs.iterrows():
    p = row["p_片側"]
    matrix.loc[row["前"], row["後"]] = f"{p:.3f} {add_stars(p)}"
matrix

### 見方
- 片側は「後の区分のほうが得点が多い」という向きだけを見ている。前の区分のほうが多い組では p値が大きくなる。
- どの区分の組み合わせを見るかを、結果を見てから選ぶと、有意な組が出やすくなってしまう。全組を見るときは、補正後の p値で判断する。

## 10. 0〜18%を細かく分けて調べる

区分1（0〜18%）が他の区分より得点が少ない、という様子だったので、区分1の中をさらに細かく分けて、同じ検定をする。
区分2〜6（18〜100%）は、そのまま使う。

細かく分けるときも、3節と同じように、**各区分の試合数がほぼそろうように**区切りをデータから決める（試合の進捗率の値ごとに試合数を数えて、累計が 1/N, 2/N, … にいちばん近い進捗率を区切りにする）。
区分1の中の分け方は `N_SUB` で変えられる。分けすぎると、1区分の得点数が少なくなって、検定で差が出にくくなる。

In [ ]:
N_SUB = 3   # 区分1を何個に分けるか

# 区分1に入る試合だけを取り出す
first_matches = match_df[match_df["progress_bin"] == BIN_ORDER[0]]

# 進捗率の値ごとに試合数を数えて、累計が 1/N_SUB, 2/N_SUB, ... にいちばん近い進捗率を区切りにする
sub_counts = first_matches["match_progress"].value_counts().sort_index()
sub_cumulative = sub_counts.cumsum()
sub_total = sub_cumulative.iloc[-1]

sub_inner_edges = []
for j in range(1, N_SUB):
    target = sub_total * j / N_SUB
    nearest = (sub_cumulative - target).abs().idxmin()
    sub_inner_edges.append(nearest)

# 区分1を細かく分けた区切り + 区分2〜6の区切り
FINE_EDGES = np.array([0.0] + sub_inner_edges + list(BIN_EDGES[1:]))
FINE_ORDER = [f"{int(round(FINE_EDGES[i] * 100))}〜{int(round(FINE_EDGES[i + 1] * 100))}%"
              for i in range(len(FINE_EDGES) - 1)]

print("区切り:", np.round(FINE_EDGES, 3))
print("区分名:", FINE_ORDER)

match_df["fine_bin"] = pd.cut(match_df["match_progress"], bins=FINE_EDGES, labels=FINE_ORDER, include_lowest=True)

区分ごとに、試合数、得点の合計、1試合あたりの得点（と95%区間）を求める。

In [ ]:
fine_summary = match_df.groupby("fine_bin", observed=False).agg(
    試合数=("match_id", "size"),
    得点の合計=("total_goals", "sum"),
)
fine_summary["1試合あたり得点"] = fine_summary["得点の合計"] / fine_summary["試合数"]
fine_summary["下限(95%)"] = chi2.ppf(0.025, 2 * fine_summary["得点の合計"]) / 2 / fine_summary["試合数"]
fine_summary["上限(95%)"] = chi2.ppf(0.975, 2 * (fine_summary["得点の合計"] + 1)) / 2 / fine_summary["試合数"]

print("試合数の合計:", int(fine_summary["試合数"].sum()) == n_matches, " 得点の合計:", int(fine_summary["得点の合計"].sum()) == len(goal_rows))
fine_summary.round(3)

グラフにする。棒の上の数字は1試合あたり得点、誤差棒は95%区間。棒の下の `n=` は試合数。

In [ ]:
n_fine = len(FINE_ORDER)
x = np.arange(n_fine)
means = fine_summary["1試合あたり得点"].to_numpy()
lower = fine_summary["下限(95%)"].to_numpy()
upper = fine_summary["上限(95%)"].to_numpy()

fig, ax = plt.subplots(figsize=(11, 6))
colors = [ "#CC6677" if i < N_SUB else BAR_COLOR for i in range(n_fine)]   # 区分1を細かく分けたものは色を変える
ax.bar(x, means, color=colors, edgecolor="white")
ax.errorbar(x, means, yerr=[means - lower, upper - means], fmt="none", ecolor="black", capsize=4)
for xi, v, u, n in zip(x, means, upper, fine_summary["試合数"]):
    ax.text(xi, u, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    ax.text(xi, 0.05, f"n={n}", ha="center", va="bottom", fontsize=9, color="white")
ax.axhline(len(goal_rows) / n_matches, color="gray", linestyle="--", linewidth=1)   # 全体の1試合あたり得点
ax.set_xticks(x)
ax.set_xticklabels(FINE_ORDER)
ax.set_ylim(0, upper.max() * 1.1)
ax.set_xlabel("リーグの進捗率")
ax.set_ylabel("1試合あたり得点（両チームの合計）")
ax.set_title("区分1を細かく分けた、1試合あたりの得点（点線は全体の平均）")
plt.tight_layout()
plt.savefig("figures/scores_per_match_fine_bins.png", dpi=150)
plt.show()

### 10.2 検定

9と同じ二項検定を、区分の組み合わせごとに行う。まず、1組を検定する関数を作る。

$$H_0:\ \bar\mu_a=\bar\mu_b,\qquad H_1:\ \bar\mu_a<\bar\mu_b$$

（$a$ が前の区分、$b$ が後の区分。$H_0$ のもとで $p_0=\dfrac{n_b}{n_a+n_b}$）

In [ ]:
def compare_two(n_a, S_a, n_b, S_b):
    # 前の区分(a)と後の区分(b)の1試合あたり得点を比べる二項検定
    S_ab = S_a + S_b
    p0_ab = n_b / (n_a + n_b)
    res = binomtest(S_b, S_ab, p0_ab)   # 両側
    ci = res.proportion_ci(confidence_level=0.95, method="exact")
    return {
        "前の1試合あたり": S_a / n_a,
        "後の1試合あたり": S_b / n_b,
        "率比 R": (S_b / S_ab) / (1 - S_b / S_ab) * n_a / n_b,
        "Rの下限": ci.low / (1 - ci.low) * n_a / n_b,
        "Rの上限": ci.high / (1 - ci.high) * n_a / n_b,
        "p_片側": binomtest(S_b, S_ab, p0_ab, alternative="greater").pvalue,
        "p_両側": res.pvalue,
    }

def holm_adjust(p_values):
    # ホルム法：小さい順に (組の数, 組の数-1, ...) 倍して、大小の順が崩れないようにする
    p_values = pd.Series(p_values).reset_index(drop=True)
    order = p_values.sort_values().index
    adjusted = pd.Series(index=p_values.index, dtype=float)
    running_max = 0
    for rank, idx in enumerate(order):
        running_max = max(running_max, min(1.0, p_values[idx] * (len(p_values) - rank)))
        adjusted[idx] = running_max
    return adjusted.to_numpy()

区分1を細かく分けた区分と、区分2〜6をまとめたもの（18〜100%）を比べる。区分1のどの部分が低いのかを見る。

In [ ]:
rest = fine_summary.iloc[N_SUB:]                 # 区分2〜6
n_rest = int(rest["試合数"].sum())
S_rest = int(rest["得点の合計"].sum())
print(f"区分2〜6をまとめたもの: 試合数 {n_rest}, 得点の合計 {S_rest}, 1試合あたり {S_rest / n_rest:.3f}")

rest_rows = []
for label in FINE_ORDER[:N_SUB]:
    row = compare_two(int(fine_summary.loc[label, "試合数"]), int(fine_summary.loc[label, "得点の合計"]), n_rest, S_rest)
    rest_rows.append({
        "比較": f"{label} → 区分2〜6",
        "前の1試合あたり": round(row["前の1試合あたり"], 3),
        "後の1試合あたり": round(row["後の1試合あたり"], 3),
        "率比 R": round(row["率比 R"], 3),
        "Rの95%区間": f"{row['Rの下限']:.3f} 〜 {row['Rの上限']:.3f}",
        "p値（片側）": f"{row['p_片側']:.4f} {add_stars(row['p_片側'])}",
        "p値（両側）": f"{row['p_両側']:.4f} {add_stars(row['p_両側'])}",
    })
pd.DataFrame(rest_rows).set_index("比較")

すべての組み合わせ（区分の数を $m$ とすると $m(m-1)/2$ 組）を検定する。多重比較の補正（ボンフェローニ、ホルム）もつける。

In [ ]:
fine_pair_rows = []
for i, k in combinations(range(n_fine), 2):
    a, b = FINE_ORDER[i], FINE_ORDER[k]
    row = compare_two(int(fine_summary.loc[a, "試合数"]), int(fine_summary.loc[a, "得点の合計"]),
                      int(fine_summary.loc[b, "試合数"]), int(fine_summary.loc[b, "得点の合計"]))
    row.update({"前": a, "後": b})
    fine_pair_rows.append(row)

fine_pairs = pd.DataFrame(fine_pair_rows)
n_fine_pairs = len(fine_pairs)
fine_pairs["p_ボンフェローニ"] = (fine_pairs["p_両側"] * n_fine_pairs).clip(upper=1)
fine_pairs["p_ホルム"] = holm_adjust(fine_pairs["p_両側"])
print(f"{n_fine_pairs} 組を検定した")

fine_pairs_show = pd.DataFrame({
    "比較": fine_pairs["前"] + " → " + fine_pairs["後"],
    "前の1試合あたり": fine_pairs["前の1試合あたり"].round(3),
    "後の1試合あたり": fine_pairs["後の1試合あたり"].round(3),
    "率比 R": fine_pairs["率比 R"].round(3),
    "p値（片側）": [f"{p:.4f} {add_stars(p)}" for p in fine_pairs["p_片側"]],
    "p値（両側）": [f"{p:.4f} {add_stars(p)}" for p in fine_pairs["p_両側"]],
    "両側・ボンフェローニ": [f"{p:.4f} {add_stars(p)}" for p in fine_pairs["p_ボンフェローニ"]],
    "両側・ホルム": [f"{p:.4f} {add_stars(p)}" for p in fine_pairs["p_ホルム"]],
}).set_index("比較")
fine_pairs_show

片側の p値を、行が前の区分、列が後の区分の表にする（$H_1$：後の区分のほうが得点が多い）。

In [ ]:
fine_matrix = pd.DataFrame("", index=FINE_ORDER, columns=FINE_ORDER)
for _, row in fine_pairs.iterrows():
    fine_matrix.loc[row["前"], row["後"]] = f"{row['p_片側']:.3f} {add_stars(row['p_片側'])}"
fine_matrix

### 注意
- `*` は10%有意、`**` は5%有意、`***` は1%有意。
- 区分1を細かく分ける位置や個数を、結果を見てから決めると、有意な組が出やすくなってしまう。`N_SUB` を変えて試すときは、補正後の p値で判断する。
- 区分1の中の区切りは、進捗率の値そのもの（何節目か）で決まるので、リーグやシーズンによって同じ区分に入る節がずれる。

## 11. 0〜18%だけを、さらに細かく分ける

ここからは、区分1（0〜18%）に入る試合だけを使う。この中を `N_EARLY` 個に分けて、区分ごとの1試合あたり得点を比べる。

分け方は3節や10節と同じで、**各区分の試合数がほぼそろうように**、進捗率の値ごとに試合数を数えて区切りを決める。
分けるほど1区分の試合数と得点数が減るので、95%区間は広くなり、検定で差が出にくくなる。

In [ ]:
N_EARLY = 6   # 0〜18%の中を何個に分けるか

early_matches = match_df[match_df["progress_bin"] == BIN_ORDER[0]].copy()   # 0〜18%の試合だけ
print(f"0〜18%の試合数: {len(early_matches)}, 得点の合計: {int(early_matches['total_goals'].sum())}")

# 進捗率の値ごとに試合数を数えて、累計が 1/N, 2/N, ... にいちばん近い進捗率を区切りにする
early_counts = early_matches["match_progress"].value_counts().sort_index()
early_cumulative = early_counts.cumsum()
early_total = early_cumulative.iloc[-1]

early_inner_edges = []
for j in range(1, N_EARLY):
    target = early_total * j / N_EARLY
    nearest = (early_cumulative - target).abs().idxmin()
    early_inner_edges.append(nearest)

EARLY_EDGES = np.array([0.0] + early_inner_edges + [BIN_EDGES[1]])
print("区切り:", np.round(EARLY_EDGES, 3))
print("区切りが重なっていないか:", len(set(np.round(EARLY_EDGES, 6))) == len(EARLY_EDGES))

# 区分名は、パーセントを小数点1桁まで書く（細かいので）
EARLY_ORDER = [f"{EARLY_EDGES[i] * 100:.1f}〜{EARLY_EDGES[i + 1] * 100:.1f}%" for i in range(N_EARLY)]
early_matches["early_bin"] = pd.cut(early_matches["match_progress"], bins=EARLY_EDGES, labels=EARLY_ORDER, include_lowest=True)

区分ごとに、試合数、得点の合計、1試合あたりの得点（と95%区間）を求める。

In [ ]:
early_summary = early_matches.groupby("early_bin", observed=False).agg(
    試合数=("match_id", "size"),
    得点の合計=("total_goals", "sum"),
)
early_summary["1試合あたり得点"] = early_summary["得点の合計"] / early_summary["試合数"]
early_summary["下限(95%)"] = chi2.ppf(0.025, 2 * early_summary["得点の合計"]) / 2 / early_summary["試合数"]
early_summary["上限(95%)"] = chi2.ppf(0.975, 2 * (early_summary["得点の合計"] + 1)) / 2 / early_summary["試合数"]

print("試合数の合計:", int(early_summary["試合数"].sum()) == len(early_matches),
      " 得点の合計:", int(early_summary["得点の合計"].sum()) == int(early_matches["total_goals"].sum()))
early_summary.round(3)

グラフにする。棒の上の数字は1試合あたり得点、誤差棒は95%区間、棒の中の `n=` は試合数。点線は、0〜18%全体の1試合あたり得点。

In [ ]:
x = np.arange(N_EARLY)
means = early_summary["1試合あたり得点"].to_numpy()
lower = early_summary["下限(95%)"].to_numpy()
upper = early_summary["上限(95%)"].to_numpy()

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x, means, color="#CC6677", edgecolor="white")
ax.errorbar(x, means, yerr=[means - lower, upper - means], fmt="none", ecolor="black", capsize=4)
for xi, v, u, n in zip(x, means, upper, early_summary["試合数"]):
    ax.text(xi, u, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    ax.text(xi, 0.05, f"n={n}", ha="center", va="bottom", fontsize=9, color="white")
ax.axhline(early_matches["total_goals"].mean(), color="gray", linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(EARLY_ORDER)
ax.set_ylim(0, upper.max() * 1.1)
ax.set_xlabel("リーグの進捗率（0〜18%の中）")
ax.set_ylabel("1試合あたり得点（両チームの合計）")
ax.set_title("0〜18%を細かく分けた、1試合あたりの得点（点線は0〜18%全体の平均）")
plt.tight_layout()
plt.savefig("figures/scores_per_match_early_bins.png", dpi=150)
plt.show()

### 11.2 検定

10.2 で作った関数 `compare_two`（二項検定）と `holm_adjust`（ホルム法）を使って、区分のすべての組み合わせ（$N$ 個なら $N(N-1)/2$ 組）を検定する。

$$H_0:\ \bar\mu_a=\bar\mu_b,\qquad H_1:\ \bar\mu_a<\bar\mu_b\quad(a\text{ が前の区分、}b\text{ が後の区分})$$

In [ ]:
early_pair_rows = []
for i, k in combinations(range(N_EARLY), 2):
    a, b = EARLY_ORDER[i], EARLY_ORDER[k]
    row = compare_two(int(early_summary.loc[a, "試合数"]), int(early_summary.loc[a, "得点の合計"]),
                      int(early_summary.loc[b, "試合数"]), int(early_summary.loc[b, "得点の合計"]))
    row.update({"前": a, "後": b})
    early_pair_rows.append(row)

early_pairs = pd.DataFrame(early_pair_rows)
n_early_pairs = len(early_pairs)
early_pairs["p_ボンフェローニ"] = (early_pairs["p_両側"] * n_early_pairs).clip(upper=1)
early_pairs["p_ホルム"] = holm_adjust(early_pairs["p_両側"])
print(f"{n_early_pairs} 組を検定した")

early_pairs_show = pd.DataFrame({
    "比較": early_pairs["前"] + " → " + early_pairs["後"],
    "前の1試合あたり": early_pairs["前の1試合あたり"].round(3),
    "後の1試合あたり": early_pairs["後の1試合あたり"].round(3),
    "率比 R": early_pairs["率比 R"].round(3),
    "p値（片側）": [f"{p:.4f} {add_stars(p)}" for p in early_pairs["p_片側"]],
    "p値（両側）": [f"{p:.4f} {add_stars(p)}" for p in early_pairs["p_両側"]],
    "両側・ボンフェローニ": [f"{p:.4f} {add_stars(p)}" for p in early_pairs["p_ボンフェローニ"]],
    "両側・ホルム": [f"{p:.4f} {add_stars(p)}" for p in early_pairs["p_ホルム"]],
}).set_index("比較")
early_pairs_show

片側の p値を、行が前の区分、列が後の区分の表にする（$H_1$：後の区分のほうが得点が多い）。

In [ ]:
early_matrix = pd.DataFrame("", index=EARLY_ORDER, columns=EARLY_ORDER)
for _, row in early_pairs.iterrows():
    early_matrix.loc[row["前"], row["後"]] = f"{row['p_片側']:.3f} {add_stars(row['p_片側'])}"
early_matrix

### 注意
- `*` は10%有意、`**` は5%有意、`***` は1%有意。
- 区切りは進捗率の値（何節目か）で決まるので、1つの区分は数節ぶんにあたる。シーズンの試合数が違うと、同じ進捗率でも節の番号が違う（38試合のシーズンの3節目は約8%、22試合のシーズンの2節目は約9%）。
- 区分の数 `N_EARLY` を変えて、結果を見てから決めると、有意な組が出やすくなってしまう。補正後の p値で判断する。